In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost==3.1.3','pandas==2.2.3','numpy==2.3.5','pyarrow==26.0.0','holidays==0.106','catboost==1.2.10','lightgbm==4.7.0'])


In [ ]:
from pathlib import Path
p=Path('/kaggle/working/stage2_launcher.py')
p.write_text('"""New original fold caches and development-only rolling CUDA experiments."""\nimport base64,gc,hashlib,io,json,os,shutil,sys,time,zipfile\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nPAYLOAD=\'...\'\n\ndef main():\n    root=Path(\'/tmp/x5_stage2_project\');parent=root/\'Forecasting\';here=parent/\'research_stage2\'\n    here.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:\n        for name in [\'features.py\',\'experiment.py\',\'plan.md\']:(here/name).write_bytes(archive.read(name))\n        for name in [\'notebook_api.py\',\'run_validation.py\']:(parent/name).write_bytes(archive.read(name))\n        stores=pd.read_parquet(io.BytesIO(archive.read(\'stores.parquet\')))\n        locations=pd.read_csv(io.BytesIO(archive.read(\'locations.csv\')))\n        old_manifest=json.loads(archive.read(\'old_manifest.json\'))\n    source_manifest=list(Path(\'/kaggle/input\').rglob(\'input_manifest.json\'))\n    research_manifest=list(Path(\'/kaggle/input\').rglob(\'research_manifest.json\'))\n    assert len(source_manifest)==len(research_manifest)==1\n    source=source_manifest[0].parent;derived=research_manifest[0].parent\n    raw_dir=root/\'Data/Shops\';raw_dir.mkdir(parents=True,exist_ok=True)\n    baseline=parent/\'baseline\';baseline.mkdir(exist_ok=True)\n    (raw_dir/\'train_2.csv\').symlink_to(source/\'train_2.csv\')\n    (baseline/\'original_solution.ipynb\').symlink_to(source/\'original_solution.ipynb\')\n    sys.path.insert(0,str(parent));sys.path.insert(0,str(here))\n    from notebook_api import load_recipe,read_raw\n    from run_validation import get_fold\n    from features import compute\n    from experiment import SCREEN,execute\n    recipe=load_recipe();raw=read_raw(recipe)\n    new,info=compute(raw,stores,locations)\n    changed=raw.copy();index=(changed.year-2023)*12+changed.month-1\n    changed.loc[index>=19,\'pto\']*=37\n    other,_=compute(changed,stores,locations)\n    pd.testing.assert_frame_equal(new.loc[new.year_month_index<=19].reset_index(drop=True),\n        other.loc[other.year_month_index<=19].reset_index(drop=True))\n    cache=here/\'cache\';cache.mkdir(exist_ok=True)\n    new.to_parquet(cache/\'features.parquet\',index=False)\n    info[\'implementation_sha256\']=hashlib.sha256((here/\'features.py\').read_bytes()).hexdigest()\n    (cache/\'feature_manifest.json\').write_text(json.dumps(info,indent=2))\n    (here/\'feature_checks.json\').write_text(json.dumps(dict(status=\'passed\',gpu_cutoff=19,\n        implementation_sha256=info[\'implementation_sha256\'],features=len(new.columns)-2),indent=2))\n    old_dir=parent/\'research/cache\';old_dir.mkdir(parents=True,exist_ok=True)\n    plan=json.loads(research_manifest[0].read_text());columns=list(dict.fromkeys(c for g in plan[\'groups\'].values() for c in g))\n    history=pd.concat([pd.read_parquet(derived/name,columns=[\'new_id\',\'year_month_index\']+columns)\n        for name in [\'2025-02_train.parquet\',\'2025-02_holdout.parquet\']],ignore_index=True)\n    assert not history.duplicated([\'new_id\',\'year_month_index\']).any()\n    history.to_parquet(old_dir/\'features.parquet\',index=False);stores.to_parquet(old_dir/\'stores.parquet\',index=False)\n    (old_dir/\'feature_manifest.json\').write_text(json.dumps(old_manifest,indent=2))\n    del raw,new,other,changed,history;gc.collect()\n    # Prepare all inputs, but do not fit/read aggregate errors for reserved months.\n    for fold in [\'2024-05\',\'2024-06\',\'2024-07\',\'2024-08\',\'2024-10\',\'2024-11\',\'2024-12\']:\n        year,month=map(int,fold.split(\'-\'));compact=get_fold(recipe,year,month);del compact;gc.collect()\n        print(\'PREPARED\',fold,flush=True)\n    output=Path(\'/kaggle/working\')\n    with zipfile.ZipFile(output/\'prepared_stage2.zip\',\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n        for p in (parent/\'cache\').rglob(\'*\'):\n            if p.is_file():archive.write(p,p.relative_to(parent/\'cache\').as_posix())\n        archive.write(cache/\'features.parquet\',\'stage2_features.parquet\')\n    print(\'PREPARED_ARCHIVE_READY\',flush=True)\n    for fold in [\'2024-08\',\'2024-06\',\'2024-07\']:\n        for config in SCREEN:execute(\'xgboost_log_diff\',fold,config,\'rolling\',threads=4)\n    for config in [\'baseline\',\'previous\']:execute(\'xgboost_log_diff\',\'2024-08\',config,\'full\',threads=4)\n    with zipfile.ZipFile(output/\'cuda_development.zip\',\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n        for folder in [here/\'results\',here/\'inner\']:\n            for p in folder.rglob(\'*\'):\n                if p.is_file():archive.write(p,p.relative_to(here).as_posix())\n    (output/\'stage2_gpu_checks.json\').write_text(json.dumps(dict(status=\'passed\',xgboost_version=\'3.1.3\',\n        outer_months=[\'2024-06\',\'2024-07\',\'2024-08\'],reserved_months_not_evaluated=[\'2024-11\',\'2024-12\'],\n        source_datasets_unchanged=True,original_folds_built_with_preserved_recipe=True,\n        outer_targets_never_used_for_stopping=True,\n        archives={p.name:dict(bytes=p.stat().st_size,sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in output.glob(\'*.zip\')}),indent=2))\n    print(\'PHASE1_DONE\',flush=True)\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
subprocess.check_call([sys.executable,'-u',str(p)])
